In [6]:
"""
IPL Win Predictor — Feature Engineering v2
===========================================
Fixes weak features by replacing 0.5-defaulted win rates with:
  - Elo ratings (never defaults, always informative)
  - Exponentially weighted recent form
  - Season-specific win rate
  - Toss-decision-venue interaction
  - Head-to-head win margin strength

Outputs: prematch_features.csv (replaces v1 output)
"""

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")


# ─────────────────────────────────────────────
# 1. LOAD & CLEAN
# ─────────────────────────────────────────────

TEAM_NAME_MAP = {
    "Delhi Daredevils": "Delhi Capitals",
    "Deccan Chargers": "Sunrisers Hyderabad",
    "Pune Warriors": "Rising Pune Supergiant",
    "Rising Pune Supergiants": "Rising Pune Supergiant",
    "Kings XI Punjab": "Punjab Kings",
}

def load_and_clean(matches_path="matches.csv"):
    df = pd.read_csv(matches_path)

    for col in ["team1", "team2", "toss_winner", "winner"]:
        df[col] = df[col].replace(TEAM_NAME_MAP)

    # Keep only completed matches
    df = df[df["result"].isin(["wickets", "runs"])].copy()
    df = df.dropna(subset=["winner"])

    df["date"] = pd.to_datetime(df["date"], dayfirst=True, errors="coerce")
    df = df.dropna(subset=["date"])
    df = df.sort_values("date").reset_index(drop=True)

    df["team1_won"]         = (df["winner"] == df["team1"]).astype(int)
    df["team1_won_toss"]    = (df["toss_winner"] == df["team1"]).astype(int)
    df["toss_winner_batted"]= (df["toss_decision"] == "bat").astype(int)

    # Ensure id column exists
    if "id" not in df.columns:
        df["id"] = df.index

    print(f"Loaded {len(df)} matches | {df['date'].min().year}–{df['date'].max().year}")
    return df


# ─────────────────────────────────────────────
# 2. ELO RATINGS
#    Starts at 1500 for every team, updates after
#    each match — never defaults to 0.5
# ─────────────────────────────────────────────

def compute_elo(df: pd.DataFrame, k=32, base=1500) -> pd.DataFrame:
    elo = {}  # team -> current elo

    t1_elo_pre, t2_elo_pre = [], []

    for _, row in df.iterrows():
        t1, t2 = row["team1"], row["team2"]

        r1 = elo.get(t1, base)
        r2 = elo.get(t2, base)

        # Expected win probabilities
        e1 = 1 / (1 + 10 ** ((r2 - r1) / 400))
        e2 = 1 - e1

        t1_elo_pre.append(round(r1, 2))
        t2_elo_pre.append(round(r2, 2))

        # Actual outcome
        s1 = row["team1_won"]
        s2 = 1 - s1

        # Update ratings
        elo[t1] = r1 + k * (s1 - e1)
        elo[t2] = r2 + k * (s2 - e2)

    df["t1_elo"]      = t1_elo_pre
    df["t2_elo"]      = t2_elo_pre
    df["elo_diff"]    = df["t1_elo"] - df["t2_elo"]   # single strong feature
    df["t1_elo_prob"] = 1 / (1 + 10 ** (-df["elo_diff"] / 400))  # P(t1 wins) per Elo

    print("Elo ratings computed.")
    return df


# ─────────────────────────────────────────────
# 3. EXPONENTIALLY WEIGHTED RECENT FORM
#    Last match counts most, older ones decay
# ─────────────────────────────────────────────

def compute_weighted_form(df: pd.DataFrame, window=8, decay=0.8) -> pd.DataFrame:
    weights = np.array([decay ** i for i in range(window)])
    weights = weights / weights.sum()  # normalize

    def get_weighted_form(team, before_idx):
        past = df.iloc[:before_idx]
        tm = past[(past["team1"] == team) | (past["team2"] == team)].tail(window)
        if len(tm) == 0:
            return 0.5
        outcomes = (tm["winner"] == team).values.astype(float)
        w = weights[-len(outcomes):]
        w = w / w.sum()
        return float(np.dot(outcomes, w))

    t1_form, t2_form = [], []
    for i in range(len(df)):
        t1_form.append(get_weighted_form(df.iloc[i]["team1"], i))
        t2_form.append(get_weighted_form(df.iloc[i]["team2"], i))

    df["t1_weighted_form"] = t1_form
    df["t2_weighted_form"] = t2_form
    df["form_diff"]        = df["t1_weighted_form"] - df["t2_weighted_form"]

    print("Weighted form computed.")
    return df


# ─────────────────────────────────────────────
# 4. SEASON-SPECIFIC WIN RATE
#    Current season form only — more relevant
#    than all-time history late in the season
# ─────────────────────────────────────────────

def compute_season_form(df: pd.DataFrame) -> pd.DataFrame:
    df["season"] = df["date"].dt.year

    t1_season_wr, t2_season_wr = [], []

    for i in range(len(df)):
        row    = df.iloc[i]
        season = row["season"]
        t1, t2 = row["team1"], row["team2"]

        past_season = df.iloc[:i][df.iloc[:i]["season"] == season]

        t1_s = past_season[(past_season["team1"] == t1) | (past_season["team2"] == t1)]
        t2_s = past_season[(past_season["team1"] == t2) | (past_season["team2"] == t2)]

        t1_season_wr.append(
            (past_season["winner"] == t1).sum() / len(t1_s) if len(t1_s) > 0 else 0.5
        )
        t2_season_wr.append(
            (past_season["winner"] == t2).sum() / len(t2_s) if len(t2_s) > 0 else 0.5
        )

    df["t1_season_wr"] = t1_season_wr
    df["t2_season_wr"] = t2_season_wr
    df["season_wr_diff"] = df["t1_season_wr"] - df["t2_season_wr"]

    print("Season win rates computed.")
    return df


# ─────────────────────────────────────────────
# 5. VENUE + TOSS FEATURES
# ─────────────────────────────────────────────

def compute_venue_toss_features(df: pd.DataFrame) -> pd.DataFrame:
    t1_venue_wr, t2_venue_wr, toss_venue_adv = [], [], []
    field_first_adv = []  # P(win) when fielding first at this venue

    for i in range(len(df)):
        row   = df.iloc[i]
        t1, t2, venue = row["team1"], row["team2"], row["venue"]
        past  = df.iloc[:i]
        v     = past[past["venue"] == venue]

        vt1 = v[(v["team1"] == t1) | (v["team2"] == t1)]
        vt2 = v[(v["team1"] == t2) | (v["team2"] == t2)]

        t1_venue_wr.append((v["winner"] == t1).sum() / len(vt1) if len(vt1) > 0 else 0.5)
        t2_venue_wr.append((v["winner"] == t2).sum() / len(vt2) if len(vt2) > 0 else 0.5)
        toss_venue_adv.append((v["winner"] == v["toss_winner"]).mean() if len(v) > 0 else 0.5)

        # Field-first advantage: P(chasing team wins) at this venue
        field_first = v[v["toss_decision"] == "field"]
        field_first_adv.append(
            (field_first["winner"] == field_first["toss_winner"]).mean()
            if len(field_first) > 0 else 0.5
        )

    df["t1_venue_wr"]    = t1_venue_wr
    df["t2_venue_wr"]    = t2_venue_wr
    df["venue_wr_diff"]  = df["t1_venue_wr"] - df["t2_venue_wr"]
    df["toss_venue_adv"] = toss_venue_adv
    df["field_first_adv"]= field_first_adv

    # Interaction: did toss winner choose to field at a field-first-favoring venue?
    df["toss_field_interact"] = df["team1_won_toss"] * (1 - df["toss_winner_batted"]) * df["field_first_adv"]

    print("Venue/toss features computed.")
    return df


# ─────────────────────────────────────────────
# 6. HEAD-TO-HEAD (proper, no leakage)
# ─────────────────────────────────────────────

def compute_h2h(df: pd.DataFrame) -> pd.DataFrame:
    h2h_wr, h2h_n = [], []

    for i in range(len(df)):
        row  = df.iloc[i]
        t1, t2 = row["team1"], row["team2"]
        past = df.iloc[:i]

        h2h = past[
            ((past["team1"] == t1) & (past["team2"] == t2)) |
            ((past["team1"] == t2) & (past["team2"] == t1))
        ]
        h2h_wr.append((h2h["winner"] == t1).sum() / len(h2h) if len(h2h) > 0 else 0.5)
        h2h_n.append(len(h2h))

    df["h2h_t1_win_rate"] = h2h_wr
    df["h2h_matches"]     = h2h_n

    print("H2H features computed.")
    return df


# ─────────────────────────────────────────────
# 7. ENCODE + BUILD FINAL DATASET
# ─────────────────────────────────────────────

FEATURES = [
    # Elo — strongest signal
    "t1_elo", "t2_elo", "elo_diff", "t1_elo_prob",
    # Form
    "t1_weighted_form", "t2_weighted_form", "form_diff",
    # Season form
    "t1_season_wr", "t2_season_wr", "season_wr_diff",
    # Venue
    "t1_venue_wr", "t2_venue_wr", "venue_wr_diff",
    "toss_venue_adv", "field_first_adv", "toss_field_interact",
    # H2H
    "h2h_t1_win_rate", "h2h_matches",
    # Toss
    "team1_won_toss", "toss_winner_batted",
    # Identity (encoded)
    "team1_enc", "team2_enc", "venue_enc", "season_enc",
]

def encode_and_build(df: pd.DataFrame):
    from sklearn.preprocessing import LabelEncoder

    le_team  = LabelEncoder()
    le_venue = LabelEncoder()

    all_teams = pd.concat([df["team1"], df["team2"]]).unique()
    le_team.fit(all_teams)
    le_venue.fit(df["venue"])

    df["team1_enc"] = le_team.transform(df["team1"])
    df["team2_enc"] = le_team.transform(df["team2"])
    df["venue_enc"] = le_venue.transform(df["venue"])
    df["season_enc"]= df["season"] - df["season"].min()

    out = df[FEATURES + ["team1_won", "date", "id", "team1", "team2", "venue"]].copy()
    out = out.dropna(subset=FEATURES)

    print(f"Final dataset: {out.shape[0]} rows, {len(FEATURES)} features")
    return out, le_team, le_venue


# ─────────────────────────────────────────────
# 8. MAIN
# ─────────────────────────────────────────────

def run_pipeline(matches_path="matches.csv"):
    print("=" * 55)

    df = load_and_clean(matches_path)
    df = compute_elo(df)
    df = compute_weighted_form(df)
    df = compute_season_form(df)
    df = compute_venue_toss_features(df)
    df = compute_h2h(df)
    df, le_team, le_venue = encode_and_build(df)

    df.to_csv("prematch_features.csv", index=False)
    print(f"Saved → prematch_features.csv")

    print(f"\nFeature signal check:")
    print(df[["elo_diff", "form_diff", "season_wr_diff", "venue_wr_diff", "h2h_t1_win_rate"]].describe().round(3).to_string())
    print(f"\nTarget balance: {df['team1_won'].mean():.3f}")

    return df, le_team, le_venue


if __name__ == "__main__":
    df, le_team, le_venue = run_pipeline("matches.csv")

Loaded 1076 matches | 2008–2024
Elo ratings computed.
Weighted form computed.
Season win rates computed.
Venue/toss features computed.
H2H features computed.
Final dataset: 1076 rows, 24 features
Saved → prematch_features.csv

Feature signal check:
       elo_diff  form_diff  season_wr_diff  venue_wr_diff  h2h_t1_win_rate
count  1076.000   1076.000        1076.000       1076.000         1076.000
mean     -1.640      0.002           0.010          0.004            0.494
std      87.755      0.288           0.345          0.417            0.218
min    -297.020     -1.000          -1.000         -1.000            0.000
25%     -58.215     -0.190          -0.167         -0.269            0.400
50%      -1.330      0.001           0.000          0.000            0.500
75%      53.075      0.203           0.177          0.310            0.600
max     279.180      1.000           1.000          1.000            1.000

Target balance: 0.509
